# Pandas — Limpeza, Tratamento e Auditoria de Dados
**UC2 — Criar e manipular dados utilizando matemática estatística**  
Prof. Cassio Ribeiro | SENAC RJ

---


## Auditoria, Limpeza e Tratamento — qual a diferença?

| | O que faz | Modifica os dados? |
|---|---|---|
| **🔍 Auditoria** | Fase de inspeção. Você não altera nada — apenas observa, mede e documenta os problemas encontrados no dataset. | ❌ Não modifica |
| **🧹 Limpeza** | Ação sobre os problemas estruturais: remove duplicatas, trata nulos, corrige tipos errados e padroniza textos. | ✅ Remove/Corrige |
| **🔧 Tratamento** | Etapa analítica: transforma os dados para análise. Trata outliers, cria colunas derivadas, normaliza valores. | ✅ Transforma |

*Na prática as três etapas se sobrepõem — mas conceitualmente seguem essa ordem.*


## Importações

In [69]:
import pandas as pd
import numpy as np

---
# 🗂️ Bloco 1 · Dataset

Dataset fictício de RH da NaraGroup — cheio de problemas intencionais.

| Coluna | Tipo esperado | Problema plantado |
|---|---|---|
| `id` | int | — |
| `nome` | string | Espaços extras: `'  Ana Lima '` |
| `departamento` | string | Categorias sujas: `'ti'`, `'TI'`, `'T.I.'`, `'Tecnologia'` |
| `cargo` | string | — |
| `salario` | float | String com vírgula: `'"3.500,00"'` · negativos · outlier 999999 |
| `data_admissao` | datetime | 3 formatos: `YYYY-MM-DD`, `DD/MM/YYYY`, `Mon-YYYY` |
| `email` | string | NaN em ~8 linhas |
| `ativo` | string | Variações: `'Sim'`, `'sim'`, `'SIM'`, `'S'`, `'1'`, `'True'` |

100 linhas · delimitador `;` · 10 linhas duplicadas · encoding UTF-8


## Primeiros passos: shape, head() e tail()

Antes de qualquer coisa, precisamos entender o que temos.  
**Sempre comece por aqui — são os olhos no dataset antes de qualquer ação.**


In [70]:
df = pd.read_csv('funcionarios_naragroup_sujo.csv', sep=';')

print(df.shape)

df.head(20)

(101, 8)


,id,nome,departamento,cargo,salario,data_admissao,email,ativo
0,32,Roberto Cardoso,FINANCEIRO,Assistente,11605.79,04/09/2022,roberto.cardoso@naragroup.com,S
1,93,Amanda Silva,Financeiro,Coordenador,10032.2,09/08/2019,amanda.silva@naragroup.com,S
2,52,Jefferson Lima,T.I.,Estagiário,7387.84,2021-06-13,jefferson.lima@naragroup.com,1
3,81,Aissa Pereira,Tecnologia,Analista,10376.08,Nov-2018,aissa.pereira@naragroup.com,True
4,7,Patrícia Rocha,FINANCEIRO,Assistente,"""14859,19""",2023-12-29,patrícia.rocha@naragroup.com,SIM
5,8,Diego Fernandes,MARKETING,Diretor,7746.46,30/07/2018,diego.fernandes@naragroup.com,Nao
6,6,Bruno Alves,Tecnologia,Assistente,6935.41,Jan-2021,bruno.alves@naragroup.com,Sim
7,69,Lívia Costa,Operações,Coordenador,9753.7,2019-12-12,lívia.costa@naragroup.com,SIM
8,25,Priscila Teixeira,vendas,Coordenador,13401.31,Aug-2022,priscila.teixeira@naragroup.com,True
9,47,Vera Andrade,FINANCEIRO,Assistente,5219.13,2022-02-04,vera.andrade@naragroup.com,S


In [71]:
df.info()  # mostra: RangeIndex, colunas, non-null count e dtype de cada coluna

<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             101 non-null    int64
 1   nome           101 non-null    str  
 2   departamento   101 non-null    str  
 3   cargo          101 non-null    str  
 4   salario        91 non-null     str  
 5   data_admissao  101 non-null    str  
 6   email          93 non-null     str  
 7   ativo          101 non-null    str  
dtypes: int64(1), str(7)
memory usage: 6.4 KB


## Radiografia do Dataset: dtypes e describe()

**dtypes** revela tipos errados. **describe()** revela outliers e distribuições suspeitas — antes de olhar os valores individualmente.  
Média e desvio padrão distorcidos no `describe()` são os primeiros sinais de outliers no dataset.


In [72]:
# Tipos de cada coluna
display(df.dtypes)

# Estatísticas das colunas numéricas
display(df.describe())

# Colunas de texto também
display(df.describe(include='object'))

id               int64
nome               str
departamento       str
cargo              str
salario            str
data_admissao      str
email              str
ativo              str
dtype: object

,id
count,101.000000
mean,50.316832
std,28.924706
min,1.000000
25%,26.000000
50%,50.000000
75%,75.000000
max,100.000000


C:\Users\santana.rodrigo\AppData\Local\Temp\ipykernel_33472\106439391.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include='object'))


,nome,departamento,cargo,salario,data_admissao,email,ativo
count,101,101,101,91,101,93,101
unique,90,31,6,82,86,83,7
top,Roberto Cardoso,FINANCEIRO,Assistente,11605.79,04/09/2022,roberto.cardoso@naragroup.com,1
freq,2,8,23,2,2,2,22


---
# 🔎 Bloco 2 · Auditoria

**Na auditoria não alteramos nada — apenas medimos e documentamos.**


## Caçando Nulos: isnull() e isnull().sum()

Colunas com mais de 15% de nulos exigem atenção especial — pode ser necessário removê-las ou reavaliar a coleta de dados.  
`df.isnull().any(axis=1)` mostra as linhas que têm pelo menos um nulo — ótimo para revisão manual.


In [73]:
# Contagem de nulos por coluna
df.isnull().sum()

id                0
nome              0
departamento      0
cargo             0
salario          10
data_admissao     0
email             8
ativo             0
dtype: int64

In [74]:
# Percentual de nulos — colunas com >15% exigem atenção especial
(df.isnull().sum() / len(df) * 100).round(2)

id               0.00
nome             0.00
departamento     0.00
cargo            0.00
salario          9.90
data_admissao    0.00
email            7.92
ativo            0.00
dtype: float64

In [75]:
# Ver QUAIS linhas têm pelo menos um nulo
df[df.isnull()     # Gera True/False para cada dado (True = nulo)
   .any(axis=1)]   # any(): existe pelo menos um True? axis=1: por linha

,id,nome,departamento,cargo,salario,data_admissao,email,ativo
11,94,Guilherme Santos,operacoes,Coordenador,7220.62,Mar-2022,NaN,1
14,96,Carla Peixoto,ti,Coordenador,NaN,Nov-2023,carla.peixoto@naragroup.com,Sim
15,53,Carla Peixoto,ti,Coordenador,NaN,Nov-2023,carla.peixoto@naragroup.com,Sim
17,70,César Gomes,OPERAÇÕES,Gerente,12852.17,2020-11-16,NaN,1
20,58,Hélio Braga,marketing,Assistente,3168.67,2022-10-18,NaN,Sim
21,46,Alexandre Lacerda,Financeiro,Estagiário,11962.92,2020-07-29,NaN,Sim
32,95,Mariana Correia,FINANCEIRO,Assistente,NaN,2022-01-21,mariana.correia@naragroup.com,1
35,34,Leandro Reis,Operações,Analista,12386.09,Nov-2018,NaN,Sim
41,22,Anderson Cruz,RH,Estagiário,11229.79,Jun-2020,NaN,1
42,77,Andréia Campos,RH,Assistente,NaN,2018-08-17,andréia.campos@naragroup.com,sim


## Caçando Duplicatas: duplicated()

**Identifique ANTES de remover** — uma duplicata pode ser um dado legítimo (ex: mesmo funcionário em duas filiais).  
Use `subset=` para verificar duplicatas apenas nas colunas realmente relevantes.

| Parâmetro | Comportamento |
|---|---|
| `keep='first'` | Marca como duplicata a partir da 2ª ocorrência (padrão) |
| `keep='last'` | Marca tudo exceto a última ocorrência |
| `keep=False` | Marca **todas** as ocorrências — use para visualizar |


In [76]:
# Quantidade de duplicatas
df.duplicated().sum()

np.int64(1)

In [77]:
# Visualizar TODAS as ocorrências duplicadas
df[df.duplicated(keep=False)]

,id,nome,departamento,cargo,salario,data_admissao,email,ativo
0,32,Roberto Cardoso,FINANCEIRO,Assistente,11605.79,04/09/2022,roberto.cardoso@naragroup.com,S
100,32,Roberto Cardoso,FINANCEIRO,Assistente,11605.79,04/09/2022,roberto.cardoso@naragroup.com,S


In [78]:
# Duplicata apenas por colunas-chave
df.duplicated(subset=['nome','departamento']).sum()

np.int64(11)

## Inconsistências de Texto: value_counts()

O mesmo dado escrito de formas diferentes gera agrupamentos errados na análise.  
Se não tratado: `df.groupby('departamento')` gerará 4 grupos separados onde deveria haver 1.


In [79]:
# Ver variações do departamento
df['departamento'].value_counts()
# Saída esperada:
# TI           12
# ti            5  ← mesmo valor!
# T.I.          3  ← mesmo valor!
# Tecnologia    4  ← mesmo valor!

departamento
FINANCEIRO                  8
vendas                      8
Tecnologia                  6
RH                          6
T.I.                        5
Operações                   5
Recursos Humanos            5
VENDAS                      5
Financeiro                  4
operacoes                   4
Fin.                        4
OPERAÇÕES                   4
marketing                   4
MARKETING                   3
marketing                   3
R.H.                        3
Oper.                       3
ti                          2
operações                   2
RH                          2
tecnologia da informação    2
financeiro                  2
Vend.                       2
financeiro                  2
rh                          1
Vendas                      1
Mkt                         1
vendas                      1
Marketing                   1
TI                          1
TI                          1
Name: count, dtype: int64

In [80]:
qtde_depart = (
    df.groupby('departamento')
    .size()
    .reset_index(name='Quantidade')
    )

qtde_depart

,departamento,Quantidade
0,FINANCEIRO,8
1,Fin.,4
2,Financeiro,4
3,MARKETING,3
4,Marketing,1
5,Mkt,1
6,OPERAÇÕES,4
7,Oper.,3
8,Operações,5
9,R.H.,3


In [81]:
# Outras colunas para checar:
print('--- ativo ---')
print(df['ativo'].value_counts())

print('\n--- cargo ---')
print(df['cargo'].value_counts())

# nome: verificar espaços invisíveis
print('\n--- nomes com espaço nas bordas ---')
mascara = df['nome'].str.startswith(' ') | df['nome'].str.endswith(' ')
print(df.loc[mascara, 'nome'].tolist())

--- ativo ---
ativo
1       22
True    18
Sim     18
Nao     12
S       11
sim     11
SIM      9
Name: count, dtype: int64

--- cargo ---
cargo
Assistente     23
Estagiário     17
Coordenador    16
Analista       16
Gerente        16
Diretor        13
Name: count, dtype: int64

--- nomes com espaço nas bordas ---
['  Jefferson Lima ', '  Diego Fernandes ', '  Cristiane Tavares ', '  Cíntia Ramos ', '  Renan Barros ', '  Letícia Moreira ', '  Aline Ferreira ', '  Débora Vieira ', '  Monique Rocha ', '  Felipe Gomes ', '  Tatiane Cunha ', '  Fernanda Costa ']


## Relatório de Auditoria — Resumo dos Problemas

Antes de agir, documente. O relatório de auditoria é o contrato entre o analista e os dados.

| Coluna / Problema | Tipo | Qtd | Severidade |
|---|---|---|---|
| `salario` | Tipo errado — string com aspas e vírgula | ~15 | 🔴 Alta |
| `data_admissao` | Tipo errado (object) + 3 formatos mistos | 100 | 🔴 Alta |
| Toda a tabela | 10 linhas completamente duplicadas | 10 | 🔴 Alta |
| `email` | Valores nulos | 8 | 🔴 Alta |
| `departamento` | Inconsistência de capitalização e variações | ~24 | 🟡 Média |
| `nome` | Espaços extras nas bordas | ~12 | 🟡 Média |
| `salario` | Valores negativos (impossíveis) | 3 | 🟡 Média |
| `salario` | Outlier extremo (999999) | 1 | 🟡 Média |
| `ativo` | Capitalização e formatos inconsistentes | var. | 🟢 Baixa |


---
# ⚠️ A Ordem Importa
**Tratar conteúdo ANTES de converter o tipo — princípio fundamental.**

```python
# ❌ ERRADO — mediana calculada com dados sujos
mediana = pd.to_numeric(df['salario'], errors='coerce').median()
# Aspas e vírgulas excluem dados da mediana!
df['salario'].fillna(mediana)

# ✅ CORRETO
# 1. Primeiro: limpar o conteúdo
df['salario'].str.replace('"','').str.replace(',','.')
# 2. Depois: converter tipo
df['salario'] = pd.to_numeric(df['salario'], errors='coerce')
# 3. Só então: calcular mediana
mediana = df['salario'].median()
```

| Passo | O que fazer |
|---|---|
| 1 | **Limpar strings** — remover aspas, trocar vírgula, tirar unidades |
| 2 | **Converter tipo** — `pd.to_numeric()` · `pd.to_datetime()` · `astype()` |
| 3 | **Calcular estatísticas** — mediana, média — agora com dados consistentes |


---
# 🧹 Bloco 3 · Limpeza


In [82]:
# Guardar shape inicial para o relatório final
shape_inicial = df.shape[0]
print(f'Registros iniciais: {shape_inicial}')

Registros iniciais: 101


## Removendo Duplicatas: drop_duplicates()

**Atenção:** nem toda linha repetida é um erro! Um funcionário pode aparecer em dois registros legítimos (ex: transferência).  
Use `subset=` para verificar duplicatas apenas nas colunas realmente relevantes.


In [83]:
# Remover duplicatas completas
df = df.drop_duplicates()
print(shape_inicial - df.shape[0])



1


In [84]:
# Duplicata por colunas específicas (quando necessário)
# df = df.drop_duplicates(
#     subset=['nome','departamento'],
#     keep='first')


## Tratando Nulos: dropna() vs fillna()

A decisão entre remover ou preencher depende do contexto — não existe regra única.

**dropna() — quando remover?**
- Coluna com > 50% de nulos
- Campo obrigatório ausente (id, email)
- Sem padrão lógico para preencher

**fillna() — quando preencher?**
- Nulos esporádicos em coluna numérica
- Categoria desconhecida → `'Não informado'`
- Dado pode ser inferido do contexto

**Preenchimento inteligente — qual medida usar?**
- **Média** → colunas numéricas contínuas e sem outliers
- **Mediana** → colunas numéricas com outliers ou assimetria
- **Moda** → colunas categóricas ou discretas


In [85]:
df = df.dropna(subset=['email'])
df = df.dropna(thresh=1)  # remove linhas com menos de 1 valores não-nulos
df

,id,nome,departamento,cargo,salario,data_admissao,email,ativo
0,32,Roberto Cardoso,FINANCEIRO,Assistente,11605.79,04/09/2022,roberto.cardoso@naragroup.com,S
1,93,Amanda Silva,Financeiro,Coordenador,10032.2,09/08/2019,amanda.silva@naragroup.com,S
2,52,Jefferson Lima,T.I.,Estagiário,7387.84,2021-06-13,jefferson.lima@naragroup.com,1
3,81,Aissa Pereira,Tecnologia,Analista,10376.08,Nov-2018,aissa.pereira@naragroup.com,True
4,7,Patrícia Rocha,FINANCEIRO,Assistente,"""14859,19""",2023-12-29,patrícia.rocha@naragroup.com,SIM
...,...,...,...,...,...,...,...,...
95,42,Willian Duarte,Recursos Humanos,Analista,6032.64,Jan-2018,willian.duarte@naragroup.com,1
96,16,Rodrigo Batista,Vend.,Gerente,13975.81,16/04/2024,rodrigo.batista@naragroup.com,True
97,99,Viviane Gomes,R.H.,Diretor,9935.7,20/05/2020,viviane.gomes@naragroup.com,1
98,3,Fernanda Costa,FINANCEIRO,Coordenador,11250.05,2024-08-07,fernanda.costa@naragroup.com,True


In [86]:


df['email'] = df['email'].fillna('sem_email')
df['salario'] = df['salario'].fillna(mediana)  # será feito após limpeza do tipo


NameError: name 'mediana' is not defined

## Corrigindo Tipos: strings, números e datas

Dados com tipo errado impedem cálculos, filtros e visualizações corretos.  
**Sempre limpar ANTES de converter.**

`salario` — limpeza antes da conversão:
```
"3.500,00"  →  str.replace(',','.')  →  "3.500.00"  →  pd.to_numeric()  →  3500.0
     str                                    str                              float
```


In [ ]:
# Passo 1: limpar conteúdo
df['salario'] = df['salario'].astype(str).str.replace('"', '').str.replace(',', '.')

# Passo 2: converter tipo
df['salario'] = pd.to_numeric(df['salario']) # pd to numeric não usa o: , errors='coerce')

print('Tipo:', df['salario'].dtype)
print('Nulos gerados:', df['salario'].isnull().sum())

Tipo: float64
Nulos gerados: 10


In [ ]:
# Datas — múltiplos formatos
# format='mixed'  → aceita múltiplos formatos de data na mesma coluna
# dayfirst=True   → interpreta DD/MM/YYYY corretamente
# errors='coerce' → datas inválidas viram NaT em vez de quebrar
df['data_admissao'] = pd.to_datetime(
    df['data_admissao'],
    format='mixed',
    dayfirst=True,
    errors='coerce')

print('Tipo:', df['data_admissao'].dtype)
print('NaT (não convertidos):', df['data_admissao'].isnull().sum())

Tipo: datetime64[us]
NaT (não convertidos): 0


In [ ]:
# Agora que salario está limpo, calcular mediana com dados consistentes
mediana_salario = df['salario'].median()
print(f'Mediana calculada com dados limpos: R$ {mediana_salario:,.2f}')
df['salario'] = df['salario'].fillna(mediana_salario)

Mediana calculada com dados limpos: R$ 7,377.91


## Padronizando Texto: str.strip(), str.upper(), map()

| Método | O que faz |
|---|---|
| `.str.strip()` | Remove espaços nas bordas |
| `.str.lower()` | Converte para minúsculas |
| `.str.upper()` | Converte para maiúsculas |
| `.str.title()` | Primeira letra de cada palavra maiúscula |
| `.str.replace()` | Substitui substring por outra |
| `.str.contains()` | Filtra linhas que contêm padrão |

`fillna(df['departamento'])` — mantém os valores que já estavam corretos e não precisavam de remapeamento.


In [ ]:
# Espaços e capitalização — sempre primeiro
df['nome'] = df['nome'].str.strip()
df['departamento'] = df['departamento'].str.strip().str.upper()

# Ver o que sobrou após padronizar
print('Valores únicos em departamento:')
print(sorted(df['departamento'].unique()))

Valores únicos em departamento:
['FIN.', 'FINANCEIRO', 'MARKETING', 'MKT', 'OPER.', 'OPERACOES', 'OPERAÇÕES', 'R.H.', 'RECURSOS HUMANOS', 'RH', 'T.I.', 'TECNOLOGIA', 'TECNOLOGIA DA INFORMAÇÃO', 'TI', 'VEND.', 'VENDAS']


In [ ]:
# Mapear variações para o valor correto
mapa = {
    'T.I.':                     'TI',
    'TECNOLOGIA':               'TI',
    'TECNOLOGIA DA INFORMAÇÃO': 'TI',
    'R.H.':                     'RH',
    'RECURSOS HUMANOS':         'RH',
    'OPERACOES':                'OPERAÇÕES',
    'OPER.':                    'OPERAÇÕES',
    'MKT':                      'MARKETING',
    'FIN.':                     'FINANCEIRO',
    'VEND.':                    'VENDAS'}

df['departamento'] = df['departamento'].map(mapa).fillna(df['departamento'])

print('Após mapeamento:')
print(sorted(df['departamento'].unique()))

Após mapeamento:
['FINANCEIRO', 'Fin.', 'Financeiro', 'MARKETING', 'Marketing', 'Mkt', 'OPERAÇÕES', 'Oper.', 'Operações', 'RH', 'RH ', 'Recursos Humanos', 'TI', 'TI ', 'Tecnologia', 'VENDAS', 'Vend.', 'Vendas', 'financeiro', 'financeiro ', 'marketing', 'marketing ', 'operacoes', 'operações ', 'rh', 'tecnologia da informação', 'ti', 'vendas', 'vendas ']


In [ ]:
# Padronizar demais colunas de texto
df['cargo'] = df['cargo'].str.strip().str.title()
df['ativo'] = df['ativo'].str.strip().str.upper()

mapa_ativo = {
    
    'S':    'SIM',
    '1':    'SIM',
    'TRUE': 'SIM',
    'NAO':  'NÃO',
    'N':    'NÃO',
    '0':    'NÃO',
    'FALSE':'NÃO',}

df['ativo'] = df['ativo'].map(mapa_ativo).fillna(df['ativo'])
df['ativo'].value_counts()

# 'SIM': 'SIM',

ativo
SIM    80
NÃO    12
Name: count, dtype: int64

---
# 🔧 Bloco 4 · Tratamento


## Valores Impossíveis: filtro com .loc e between()

Dados que passaram pela validação do sistema mas não fazem sentido no mundo real.

**Sempre registre quantos registros foram removidos em cada etapa** — é parte da auditoria e deve ser documentado.  
`between(a,b)` é equivalente a `(df['col'] >= a) & (df['col'] <= b)` — mais legível.


In [ ]:
# Identificar salários negativos
sal_negativos = ((df["salario"]< 0).sum())
df_negativos = df.loc[(df["salario"] < 0)][['nome','departamento','salario']]
print(f'Salários negativos: {sal_negativos}')
df_negativos




#print(f'Salários negativos: {(df["salario"] < 0).sum()}')
#display(df[df['salario'] < 0][['nome','departamento','salario']])

TypeError: '<' not supported between instances of 'str' and 'int'

In [ ]:
# Remover salários negativos ou zerados
antes = df.shape[0]
df = df[df['salario'] > 0]
print(f'Removidos: {antes - df.shape[0]}')

# Filtrar dentro de um intervalo válido
antes2 = df.shape[0]
df = df[df['salario'].between(1400, 2500)]  # salário mínimo → teto realista
print(f'Removidos (fora do intervalo): {antes2 - df.shape[0]}')
print(f'Shape atual: {df.shape}')

TypeError: '>' not supported between instances of 'str' and 'int'

## Detectando e Tratando Outliers: IQR

**O que fazer com outliers? Três opções:**

| Decisão | Quando usar |
|---|---|
| **Remover** | Valor claramente inválido (ex: salário 999999 — erro de digitação) |
| **Substituir** | Registro válido, mas valor improvável — substitua pela mediana |
| **Manter** | Valor real e relevante (ex: salário de diretor — cliente VIP) |


In [ ]:
# O dropna() antes do np.array é necessário
# pois o NumPy não lida bem com nulos
dados = np.array(df['salario'].dropna())

q1  = np.percentile(dados, 25)
q3  = np.percentile(dados, 75)
iqr = q3 - q1
lim_sup = q3 + (1.5 * iqr)
lim_inf = q1 - (1.5 * iqr)

print(f'Q1:              R$ {q1:,.2f}')
print(f'Q3:              R$ {q3:,.2f}')
print(f'IQR:             R$ {iqr:,.2f}')
print(f'Limite superior: R$ {lim_sup:,.2f}')
print(f'Limite inferior: R$ {lim_inf:,.2f}')

Q1:              R$ 5,479.91
Q3:              R$ 10,118.17
IQR:             R$ 4,638.26
Limite superior: R$ 17,075.55
Limite inferior: R$ -1,477.47


In [ ]:
# Identificar os outliers
outliers = df.loc[df['salario'] > lim_sup]
print(f'Outliers encontrados: {len(outliers)}')
display(outliers[['nome','departamento','cargo','salario']])

Outliers encontrados: 0


,nome,departamento,cargo,salario


In [ ]:
# Decisão: REMOVER — valor 999999 é claramente erro de digitação
antes_out = df.shape[0]
df = df[df['salario'] <= lim_sup]
print(f'Removidos: {antes_out - df.shape[0]}')
print(f'Shape atual: {df.shape}')

# Substituir (alternativa):
# df.loc[df['salario'] > lim_sup, 'salario'] = df['salario'].median()

# Manter (alternativa):
# Documentar o outlier e manter no dataset

Removidos: 0
Shape atual: (88, 8)


## Colunas Derivadas: apply() e Operações Vetorizadas

Criar novas colunas a partir dos dados existentes enriquece o dataset para análise.  
**Operações vetorizadas (sem apply) são 10–100× mais rápidas** — use sempre que possível.  
Use `apply()` só quando precisar de lógica condicional.


In [ ]:
# Operação vetorizada — mais rápida
df['anos_empresa'] = (
    pd.Timestamp.now() - df['data_admissao']).dt.days // 365

display(df[['nome','data_admissao','anos_empresa']].head(5))

,nome,data_admissao,anos_empresa
0,Roberto Cardoso,2022-09-04,3
1,Amanda Silva,2019-08-09,6
2,Jefferson Lima,2021-06-13,4
3,Aissa Pereira,2018-11-01,7
4,Patrícia Rocha,2023-12-29,2


In [ ]:
# apply() com função — mais flexível para lógica condicional
def faixa_salarial(sal):
    if sal < 3000: return 'Júnior'
    elif sal < 8000: return 'Pleno'
    else: return 'Sênior'

df['faixa_salarial'] = df['salario'].apply(faixa_salarial)

# Resultado — novas colunas:
display(df[['nome','salario','anos_empresa','faixa_salarial']].head(8))

,nome,salario,anos_empresa,faixa_salarial
0,Roberto Cardoso,11605.79,3,Sênior
1,Amanda Silva,10032.20,6,Sênior
2,Jefferson Lima,7387.84,4,Pleno
3,Aissa Pereira,10376.08,7,Sênior
4,Patrícia Rocha,14859.19,2,Sênior
5,Diego Fernandes,7746.46,7,Pleno
6,Bruno Alves,6935.41,5,Pleno
7,Lívia Costa,9753.70,6,Sênior


---
# ✅ Bloco 5 · Resultado Final


## Dataset Antes vs. Depois — Comparativo Final

| Problema Identificado | Solução Aplicada | Etapa |
|---|---|---|
| 10 linhas duplicadas | `drop_duplicates()` | Limpeza |
| 8 nulos em email | `dropna(subset=['email'])` | Limpeza |
| `salario` como string com aspas e vírgula | `str.replace()` + `pd.to_numeric()` | Limpeza |
| `data_admissao` com 3 formatos mistos | `pd.to_datetime(format='mixed')` | Limpeza |
| `departamento` com 5+ variações | `str.strip()` + `str.upper()` + `map()` | Limpeza |
| `nome` com espaços invisíveis | `str.strip()` | Limpeza |
| 3 salários negativos | filtro `.loc > 0` | Tratamento |
| 1 outlier extremo (999999) | IQR — remoção | Tratamento |
| Sem coluna `faixa_salarial` | `apply(faixa_salarial)` | Tratamento |


In [87]:
print('=' * 48)
print('      RELATÓRIO FINAL DE AUDITORIA')
print('=' * 48)
print(f'Registros iniciais:     {shape_inicial:>6}')
print(f'Registros finais:       {df.shape[0]:>6}')
print(f'Registros removidos:    {shape_inicial - df.shape[0]:>6}')
print(f'Colunas originais:      {8:>6}')
print(f'Colunas finais:         {df.shape[1]:>6}')
print('=' * 48)
print(f'Nulos restantes:        {df.isnull().sum().sum():>6}')
print(f'Duplicatas restantes:   {df.duplicated().sum():>6}')
print('=' * 48)

      RELATÓRIO FINAL DE AUDITORIA
Registros iniciais:        101
Registros finais:           92
Registros removidos:         9
Colunas originais:           8
Colunas finais:              8
Nulos restantes:            10
Duplicatas restantes:        0


In [ ]:
display(df.head(10))

,id,nome,departamento,cargo,salario,data_admissao,email,ativo,anos_empresa,faixa_salarial
0,32,Roberto Cardoso,FINANCEIRO,Assistente,11605.79,2022-09-04,roberto.cardoso@naragroup.com,SIM,3,Sênior
1,93,Amanda Silva,FINANCEIRO,Coordenador,10032.20,2019-08-09,amanda.silva@naragroup.com,SIM,6,Sênior
2,52,Jefferson Lima,TI,Estagiário,7387.84,2021-06-13,jefferson.lima@naragroup.com,SIM,4,Pleno
3,81,Aissa Pereira,TI,Analista,10376.08,2018-11-01,aissa.pereira@naragroup.com,SIM,7,Sênior
4,7,Patrícia Rocha,FINANCEIRO,Assistente,14859.19,2023-12-29,patrícia.rocha@naragroup.com,SIM,2,Sênior
5,8,Diego Fernandes,MARKETING,Diretor,7746.46,2018-07-30,diego.fernandes@naragroup.com,NÃO,7,Pleno
6,6,Bruno Alves,TI,Assistente,6935.41,2021-01-01,bruno.alves@naragroup.com,SIM,5,Pleno
7,69,Lívia Costa,OPERAÇÕES,Coordenador,9753.70,2019-12-12,lívia.costa@naragroup.com,SIM,6,Sênior
8,25,Priscila Teixeira,VENDAS,Coordenador,13401.31,2022-08-01,priscila.teixeira@naragroup.com,SIM,3,Sênior
9,47,Vera Andrade,FINANCEIRO,Assistente,5219.13,2022-04-02,vera.andrade@naragroup.com,SIM,4,Pleno


In [ ]:
df.to_csv('funcionarios_naragroup_clean.csv', sep=';', index=False, encoding='utf-8')
print('✅ Base exportada: funcionarios_naragroup_clean.csv')

✅ Base exportada: funcionarios_naragroup_clean.csv


## Checklist de Auditoria e Limpeza

Use este checklist em qualquer projeto de análise de dados com Pandas.

### Auditoria
- [ ] `df.shape` → quantas linhas e colunas?
- [ ] `df.info()` → tipos corretos? há nulos?
- [ ] `df.describe()` → outliers numéricos?
- [ ] `df.isnull().sum()` → quais colunas têm nulos?
- [ ] `df.duplicated().sum()` → linhas repetidas?
- [ ] `df['col'].value_counts()` → inconsistências de texto?

### Limpeza & Tratamento
- [ ] Remover duplicatas (verificar contexto antes!)
- [ ] Limpar strings **ANTES** de converter tipos
- [ ] Converter tipos: `astype`, `to_numeric`, `to_datetime`
- [ ] Padronizar texto: `strip`, `upper/lower`, `map()`
- [ ] Tratar nulos com critério: `dropna` ou `fillna`
- [ ] Tratar outliers: remover, substituir ou manter
- [ ] Criar colunas derivadas para enriquecer análise
